# WRDNet Pipeline Test Notebook
**Quick verification that the full pipeline runs on Google Colab GPU**

This notebook does NOT do full training. It:
1. Sets up the environment
2. Loads data from Google Drive
3. Runs 1 epoch with batch_size=2 on a few batches
4. Verifies forward pass, loss computation, and backward pass all work

**Expected runtime: ~10 minutes** (setup + 5 training batches)

## Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

## Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!rm -rf /content/object_detection
!git clone https://github.com/soham-kar/object_detection.git /content/object_detection

# Verify clone succeeded
import os
if os.path.exists('/content/object_detection/src'):
    print('Repository cloned successfully.')
else:
    print('ERROR: Clone failed! Check your internet connection.')
    print('Trying again...')
    !git clone https://github.com/soham-kar/object_detection.git /content/object_detection

In [ ]:
# Install dependencies
import os
if not os.path.exists('/content/object_detection'):
    print('ERROR: /content/object_detection not found! Re-run the clone cell above.')
else:
    %cd /content/object_detection
    !pip install ultralytics>=8.3.0 timm>=1.0.27 opencv-python-headless \
        pycocotools tensorboard thop scipy pyyaml tqdm matplotlib seaborn \
        albumentations 2>&1 | tail -5
    print('Dependencies installed.')

In [ ]:
# Clone DehazeFormer (external dependency)
!git clone https://github.com/IDKiro/DehazeFormer.git /content/object_detection/external/DehazeFormer
%cd /content/object_detection/external/DehazeFormer
!pip install -e . 2>&1 | tail -3
%cd /content/object_detection
print('DehazeFormer installed.')

## Step 3: Mount Google Drive & Link Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
import os
import shutil

# Your data is directly in MyDrive/object_detection/ (no data/ subfolder)
DRIVE_DATA = '/content/drive/MyDrive/object_detection'
PROJECT_DATA = '/content/object_detection/data'

# Remove existing data dir and create symlink
if os.path.exists(PROJECT_DATA) and os.path.islink(PROJECT_DATA):
    os.unlink(PROJECT_DATA)
elif os.path.exists(PROJECT_DATA):
    shutil.rmtree(PROJECT_DATA)
os.symlink(DRIVE_DATA, PROJECT_DATA)

# Verify data is accessible
print('Data directories:')
for d in sorted(os.listdir(PROJECT_DATA)):
    full = os.path.join(PROJECT_DATA, d)
    if os.path.isdir(full):
        print(f'  {d}/')
    else:
        print(f'  {d}')

In [ ]:
# Check if labels exist (needed for training)
cityscapes_labels = '/content/object_detection/data/cityscapes/labels'
acdc_labels = '/content/object_detection/data/acdc_labels'

cs_ok = os.path.exists(cityscapes_labels) and len(os.listdir(cityscapes_labels)) > 0
acdc_ok = os.path.exists(acdc_labels) and len(os.listdir(acdc_labels)) > 0

print(f'Cityscapes labels: {"FOUND" if cs_ok else "MISSING"}')
print(f'ACDC labels: {"FOUND" if acdc_ok else "MISSING"}')

if not cs_ok:
    print('\nGenerating Cityscapes labels...')
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split train 2>&1 | tail -5
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split val 2>&1 | tail -5

if not acdc_ok:
    print('\nGenerating ACDC labels...')
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_train_gt_detection.json \
        --output data/acdc_labels/train 2>&1 | tail -5
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_val_gt_detection.json \
        --output data/acdc_labels/val 2>&1 | tail -5

print('\nLabels ready.')

## Step 4: Verify Data Pipeline

Test that all 4 datasets load correctly with the right shapes.

In [ ]:
%cd /content/object_detection
!python scripts/verify_data_pipeline.py 2>&1 | tail -30

## Step 5: Build Model & Verify Forward Pass

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet

config = load_config('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
print('Building WRDNet...')
model = WRDNet(config).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Parameters: {n_params:.2f}M')

# Test forward pass with dummy input
x = torch.randn(1, 3, 640, 640, device=device)
with torch.no_grad():
    outputs = model(x, return_alpha=True)

print(f'  Restored: {outputs["restored"].shape}')
det = outputs['detections']
if isinstance(det, tuple):
    print(f'  Detections: {det[0].shape}')
for k, v in outputs['alpha_maps'].items():
    print(f'  Alpha {k}: {v.shape}')
print('Forward pass OK!')

## Step 6: Smoke Test — 5 Training Batches

Run 5 batches through the full training loop (forward + loss + backward + optimizer step).
This verifies the entire pipeline works end-to-end.

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.training.losses import WRDNetLoss
from src.data.dataset import build_dataloaders

# Config for smoke test
config = load_config('configs/default.yaml')
config.batch_size = 2
config.num_workers = 2
config.epochs = 1
config.use_fda = False
config.use_dct_align = False
config.use_fsg_consistency = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
print('Building WRDNet...')
model = WRDNet(config).to(device)
print(f'  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

# Build loss
print('Building loss function...')
criterion = WRDNetLoss(config, yolo_model=model.yolo.model)

# Build dataloaders
print('Building dataloaders...')
train_loader, val_loader = build_dataloaders(config)
print(f'  Train: {len(train_loader)} batches')
print(f'  Val: {len(val_loader)} batches')

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print('\n' + '='*60)
print('SMOKE TEST: Running 5 training batches')
print('='*60)

model.train()

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 5:
        break
    
    # Move to device
    if 'synth' in batch:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch['synth'].items()}
        real = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                for k, v in batch['real'].items()}
    else:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        real = None
    
    # Forward
    outputs = model.forward_train(synth, real)
    
    # Loss
    losses = criterion(outputs, synth)
    
    # Backward
    optimizer.zero_grad()
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    loss_str = ' '.join([f'{k}={v.item():.3f}' for k, v in losses.items() if isinstance(v, torch.Tensor)])
    print(f'  Batch {batch_idx}: {loss_str}')

print('\n' + '='*60)
print('SMOKE TEST PASSED! Pipeline works end-to-end.')
print('='*60)

## Step 7: Quick 1-Epoch Training (Supervised Only)

Run 1 full epoch on Cityscapes (no domain adaptation) to verify training converges.
This uses a small subset to keep it fast (~5 minutes on T4).

In [ ]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.training.losses import WRDNetLoss
from src.data.dataset import build_dataloaders
from tqdm import tqdm

# Config for 1-epoch test — small batch to avoid OOM on T4
config = load_config('configs/default.yaml')
config.batch_size = 2
config.num_workers = 2
config.epochs = 1
config.use_fda = False
config.use_dct_align = False
config.use_fsg_consistency = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build everything
model = WRDNet(config).to(device)
criterion = WRDNetLoss(config, yolo_model=model.yolo.model)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_loader, val_loader = build_dataloaders(config)
print(f'Train: {len(train_loader)} batches, Val: {len(val_loader)} batches')

MAX_BATCHES = 20
print(f'\nTraining 1 epoch (max {MAX_BATCHES} batches, batch_size=2)...')

model.train()
epoch_losses = []
det_losses = []

pbar = tqdm(enumerate(train_loader), total=min(MAX_BATCHES, len(train_loader)), desc='Epoch 1')
for batch_idx, batch in pbar:
    if batch_idx >= MAX_BATCHES:
        break

    # Move to device
    if 'synth' in batch:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch['synth'].items()}
        real = {k: v.to(device) if isinstance(v, torch.Tensor) else
                [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                for k, v in batch['real'].items()}
    else:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        real = None

    # Forward + loss + backward
    outputs = model.forward_train(synth, real)
    losses = criterion(outputs, synth)

    optimizer.zero_grad()
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss = losses['total'].item()
    det_loss = losses['det'].item() if isinstance(losses['det'], torch.Tensor) else 0.0
    rest_loss = losses['rest'].item() if isinstance(losses['rest'], torch.Tensor) else 0.0
    epoch_losses.append(total_loss)
    det_losses.append(det_loss)

    pbar.set_postfix({'loss': f"{total_loss:.3f}", 'det': f"{det_loss:.3f}", 'rest': f"{rest_loss:.3f}"})

    # Free memory
    del outputs, losses
    torch.cuda.empty_cache()

print(f'\nEpoch 1 complete! Processed {len(epoch_losses)} batches.')
print(f'  Avg total loss: {sum(epoch_losses)/len(epoch_losses):.4f}')
print(f'  Avg det loss:   {sum(det_losses)/len(det_losses):.4f}')
print(f'  First batch:    total={epoch_losses[0]:.4f}, det={det_losses[0]:.4f}')
print(f'  Last batch:     total={epoch_losses[-1]:.4f}, det={det_losses[-1]:.4f}')
if det_losses[0] > 0:
    print(f'  Detection loss is NON-ZERO — YOLO loss working!')
else:
    print(f'  WARNING: Detection loss is 0 — YOLO loss still has device issue')
print('Training loop works end-to-end!')

## Step 8: Validation Test

Run validation on ACDC val set to verify the model can produce detections.

In [ ]:
print('Running validation on ACDC val set...')

model.eval()
val_losses = []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        if batch_idx >= 10:  # Just 10 batches for quick test
            break
        
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else 
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        
        outputs = model.forward_train(batch)
        losses = criterion(outputs, batch)
        val_losses.append(losses['total'].item())

print(f'Validation (10 batches):')
print(f'  Avg loss: {sum(val_losses)/len(val_losses):.4f}')
print('Validation OK!')

## Step 9: Summary

If all steps passed, the pipeline is ready for full training.

In [ ]:
print('='*60)
print('PIPELINE TEST SUMMARY')
print('='*60)
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  Model: WRDNet ({sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params)')
print(f'  Smoke test: 5 batches PASSED')
print(f'  1-epoch training: {len(epoch_losses)} batches PASSED')
print(f'  Validation: {len(val_losses)} batches PASSED')
print(f'  Training loss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}')
print('='*60)
print('\nEverything works! Ready for full training.')
print('To run full training, use the WRDNet_Training_Colab.ipynb notebook.')